# Visual Anagrams — Batch Worker

This notebook polls Google Drive for queued jobs from the local web app and generates 1024×1024 rotate-180 visual anagrams.

**Requirements:** Colab Pro (A100 + High-RAM), Hugging Face access to DeepFloyd IF.

The local app uses Google’s `drive.file` scope and **pre-creates** queue/result files.
This notebook should only **overwrite** those files (especially `image_1024.png`), not create differently named outputs.

**Drive layout used:**
```
My Drive/
├── visual_anagrams/
│   ├── job_queue.json
│   ├── secrets.json
│   └── colab_heartbeat.json
└── visual_anagrams_results/
    └── {job_id}/
        └── image_1024.png   # overwrite the app-created placeholder
```

Set runtime to A100 + High-RAM, run all cells, then leave the final loop cell running.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies


In [ ]:
!pip install -q -U --no-cache-dir \
  diffusers==0.35.1 \
  transformers==4.55.4 \
  huggingface-hub==0.34.4 \
  safetensors==0.7.0 \
  sentencepiece==0.2.0 \
  accelerate==1.10.1 \
  bitsandbytes==0.49.2 \
  einops==0.7.0 \
  mediapy==1.2.0

!pip install -q --no-cache-dir --no-deps --force-reinstall \
  git+https://github.com/dangeng/visual_anagrams.git

## 3. Hugging Face login

Reads `visual_anagrams/secrets.json` written by the local app (no need to paste a token here).


In [ ]:
import json
from pathlib import Path
from huggingface_hub import login

secrets_path = Path('/content/drive/MyDrive/visual_anagrams/secrets.json')
if not secrets_path.exists():
    raise FileNotFoundError(
        f'Missing {secrets_path}. Open the local app, save your Hugging Face token, '
        'and Login with Google so it syncs to Drive.'
    )

secrets = json.loads(secrets_path.read_text())
HF_TOKEN = secrets.get('huggingface_token')
if not HF_TOKEN:
    raise ValueError('secrets.json has no huggingface_token')

login(token=HF_TOKEN)
print('Hugging Face login OK')


## 4. Paths & helpers


In [ ]:
import json
import os
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
QUEUE_DIR = DRIVE_ROOT / "visual_anagrams"
RESULTS_DIR = DRIVE_ROOT / "visual_anagrams_results"
QUEUE_PATH = QUEUE_DIR / "job_queue.json"
HEARTBEAT_PATH = QUEUE_DIR / "colab_heartbeat.json"
LOCAL_RESULTS = Path("/content/va_results")

QUEUE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

if not QUEUE_PATH.exists():
    QUEUE_PATH.write_text(json.dumps({"jobs": []}, indent=2))
    print(f"Created empty queue at {QUEUE_PATH}")
else:
    print(f"Queue found at {QUEUE_PATH}")


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def read_queue():
    data = json.loads(QUEUE_PATH.read_text())
    if isinstance(data, list):
        return {"jobs": data}
    data.setdefault("jobs", [])
    return data


def write_queue(queue):
    # Atomic-ish write for Drive
    tmp = QUEUE_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(queue, indent=2))
    tmp.replace(QUEUE_PATH)


def write_heartbeat(extra=None):
    payload = {
        "last_seen": utc_now(),
        "status": "running",
        "host": "colab",
    }
    if extra:
        payload.update(extra)
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))


def update_job(queue, job_id, **fields):
    for job in queue["jobs"]:
        if job["id"] == job_id:
            job.update(fields)
            job["updated_at"] = utc_now()
            return job
    return None

## 5. Clone repo for generate.py (CLI)


In [ ]:
import os
os.chdir("/content")
!rm -rf visual_anagrams_repo
!git clone --depth 1 https://github.com/dangeng/visual_anagrams.git visual_anagrams_repo
REPO = Path("/content/visual_anagrams_repo")
print("Repo ready:", REPO)

## 6. Job runner

Runs `generate.py` with identity + rotate_180, copies `sample_1024.png` to Drive as `image_1024.png`.


In [ ]:
def run_job(job):
    job_id = job["id"]
    prompt_1 = job["prompt_1"]
    prompt_2 = job["prompt_2"]
    seed = job.get("seed", 0)
    out_dir = RESULTS_DIR / job_id
    out_dir.mkdir(parents=True, exist_ok=True)

    # App pre-creates image_1024.png under drive.file scope — overwrite it,
    # do not create a differently named file.
    dest = out_dir / "image_1024.png"

    cmd = [
        "python", "generate.py",
        "--name", job_id,
        "--save_dir", str(LOCAL_RESULTS),
        "--prompts", prompt_1, prompt_2,
        "--views", "identity", "rotate_180",
        "--num_samples", "1",
        "--num_inference_steps", "30",
        "--guidance_scale", "10.0",
        "--generate_1024",
        "--seed", str(seed),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)

    src = LOCAL_RESULTS / job_id / "0000" / "sample_1024.png"
    if not src.exists():
        matches = list((LOCAL_RESULTS / job_id).rglob("sample_1024.png"))
        if not matches:
            raise FileNotFoundError(f"No sample_1024.png found for {job_id}")
        src = matches[0]

    shutil.copy2(src, dest)
    print(f"Overwrote {dest}")
    return str(dest)


## 7. Poll loop (leave this running)

Polls every 30 seconds. Writes a heartbeat so the local app can show online status.


In [ ]:
POLL_SECONDS = 30

print("Batch worker started. Ctrl+C / interrupt to stop.")
write_heartbeat({"message": "worker_started"})

while True:
    try:
        write_heartbeat()
        queue = read_queue()
        pending = [j for j in queue["jobs"] if j.get("status") == "pending"]
        print(f"[{utc_now()}] pending={len(pending)} total={len(queue['jobs'])}")

        for job in pending:
            job_id = job["id"]
            print(f"Processing {job_id}: {job['prompt_1']} ↔ {job['prompt_2']}")
            update_job(queue, job_id, status="processing")
            write_queue(queue)
            write_heartbeat({"current_job": job_id})

            try:
                run_job(job)
                update_job(
                    queue,
                    job_id,
                    status="completed",
                    completed_at=utc_now(),
                    error_message=None,
                    image_path="image_1024.png",
                )
            except Exception as e:
                print(f"Job {job_id} failed:", e)
                update_job(
                    queue,
                    job_id,
                    status="failed",
                    error_message=str(e),
                )

            write_queue(queue)
            write_heartbeat({"last_job": job_id, "last_job_status": job.get("status")})

    except Exception as loop_err:
        print("Loop error:", loop_err)
        write_heartbeat({"status": "error", "error": str(loop_err)})

    time.sleep(POLL_SECONDS)